# ML Model Benchmarking - Analysis Notebook

This notebook provides an interactive walkthrough of the benchmarking results.
Run the full pipeline first with `python main.py` (or `python main.py --quick` for a fast run),
then use this notebook to explore and analyze the results.

In [ ]:
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

sns.set_theme(style='whitegrid', font_scale=1.1)

RESULTS_DIR = '../results'
print('Results directory:', os.path.abspath(RESULTS_DIR))

## 1. System Information

In [ ]:
from src.utils import get_device_info

device_info = get_device_info()
for key, value in device_info.items():
    print(f'{key:>20s}: {value}')

## 2. Training Results Comparison

In [ ]:
# Load all training results
training_files = {
    'PyTorch CNN': 'pytorch_cnn_results.json',
    'PyTorch MLP': 'pytorch_mlp_results.json',
    'TF CNN': 'tensorflow_cnn_results.json',
    'TF MLP': 'tensorflow_mlp_results.json',
}

training_results = {}
for name, fname in training_files.items():
    path = os.path.join(RESULTS_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            training_results[name] = json.load(f)
        print(f'Loaded: {name}')
    else:
        print(f'Not found: {path}')

# Build comparison table
rows = []
for name, data in training_results.items():
    m = data['test_metrics']
    rows.append({
        'Model': name,
        'Accuracy': f"{m['accuracy']:.4f}",
        'F1 Score': f"{m['f1_score']:.4f}",
        'AUC-ROC': f"{m['auc_roc']:.4f}",
        'Parameters': f"{data['num_parameters']:,}",
        'Train Time (s)': f"{data['training_time_ms']/1000:.1f}",
    })

if rows:
    df = pd.DataFrame(rows)
    display(df)

## 3. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, data in training_results.items():
    history = data.get('history', {})
    if not history:
        continue
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0].plot(epochs, history['train_loss'], marker='.', label=name)
    axes[1].plot(epochs, history['train_accuracy'], marker='.', label=name)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Training Loss'); axes[0].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].set_title('Training Accuracy'); axes[1].legend()
plt.tight_layout()
plt.show()

## 4. Baseline Comparison

In [ ]:
baseline_path = os.path.join(RESULTS_DIR, 'baseline_results.json')
if os.path.exists(baseline_path):
    with open(baseline_path) as f:
        baselines = json.load(f)
    
    rows = []
    for b in baselines:
        rows.append({
            'Model': b['model_name'],
            'Accuracy': f"{b['metrics']['accuracy']:.4f}",
            'F1 Score': f"{b['metrics']['f1_score']:.4f}",
            'AUC-ROC': f"{b['metrics']['auc_roc']:.4f}",
            'Train Time (ms)': f"{b['training_time_ms']:.0f}",
            'Inference Time (ms)': f"{b['inference_time_ms']:.0f}",
        })
    df_baselines = pd.DataFrame(rows)
    display(df_baselines)
else:
    print('Baseline results not found. Run: python -m src.baselines')

## 5. Inference Latency Analysis

In [ ]:
bench_path = os.path.join(RESULTS_DIR, 'inference_benchmarks.json')
if os.path.exists(bench_path):
    with open(bench_path) as f:
        bench_data = json.load(f)
    
    df_bench = pd.DataFrame(bench_data['benchmarks'])
    print(f"Total benchmark configs: {len(df_bench)}")
    print(f"Frameworks: {df_bench['framework'].unique()}")
    print(f"Devices: {df_bench['device'].unique()}")
    display(df_bench.head(10))
    
    # Latency heatmap
    for model_type in ['cnn', 'mlp']:
        subset = df_bench[df_bench['model_type'] == model_type]
        pivot = subset.pivot_table(
            index='batch_size',
            columns=['framework', 'device'],
            values='mean_latency_ms'
        )
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
        ax.set_title(f'{model_type.upper()} Inference Latency (ms)')
        plt.tight_layout()
        plt.show()
else:
    print('Benchmark results not found. Run: python -m src.benchmark')

## 6. GPU Profiling Results

In [ ]:
gpu_path = os.path.join(RESULTS_DIR, 'gpu_profiling_results.json')
if os.path.exists(gpu_path):
    with open(gpu_path) as f:
        gpu_data = json.load(f)
    
    print('=== Device Info ===')
    for k, v in gpu_data['device_info'].items():
        print(f'  {k}: {v}')
    
    if gpu_data['memory_profiles']:
        print('\n=== Memory Profiles ===')
        df_mem = pd.DataFrame(gpu_data['memory_profiles'])
        display(df_mem)
    
    if gpu_data['kernel_timing']:
        print('\n=== Kernel Timing ===')
        df_kernel = pd.DataFrame(gpu_data['kernel_timing'])
        display(df_kernel)
    
    if gpu_data['bandwidth_estimates']:
        print('\n=== Bandwidth Estimates ===')
        df_bw = pd.DataFrame(gpu_data['bandwidth_estimates'])
        display(df_bw)
else:
    print('GPU profiling results not found. Run: python -m src.gpu_profiler')
    print('(Requires NVIDIA GPU with CUDA toolkit)')

## 7. Summary & Conclusions

### Expected Observations:

1. **CNN vs MLP**: CNNs achieve higher accuracy due to spatial feature extraction via convolution layers, while MLPs treat each pixel independently.

2. **GPU Speedup**: GPU inference is significantly faster than CPU for larger batch sizes. The speedup increases with batch size until GPU memory becomes the bottleneck.

3. **Framework Comparison**: PyTorch and TensorFlow show similar performance when architectures are matched, with minor differences due to backend optimizations.

4. **Memory-Bound vs Compute-Bound**: MLP models are typically more memory-bound (large weight matrices for fully-connected layers), while CNNs are more compute-bound (convolution kernel operations).

5. **Baseline Context**: Traditional ML models (especially Random Forest) can be competitive on flattened pixel features but lack the representational capacity of deep learning models for image tasks.